# NYC Yellow Taxi Trip Analysis

## Project Overview

This project aims to perform an end-to-end data science workflow on the NYC Yellow Taxi Trip dataset. The analysis includes data understanding, data cleaning, exploratory data analysis (EDA), feature engineering, preprocessing, machine learning modeling, and model evaluation.

The dataset used in this project is publicly provided by the **New York City Taxi and Limousine Commission (TLC)**.

> **Dataset Scope:** Only the **January 2026** Yellow Taxi Trip Records are used in this project. This subset contains millions of real-world taxi trips and is sufficient for comprehensive analysis while remaining computationally manageable on a local machine.

> Data source : https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
import os

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 100
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

### Load the dataset

In [2]:
df = pd.read_parquet('data/yellow_tripdata_2026-01.parquet')
df

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.20,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.90,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.70,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.70,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.50,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3724884,2,2026-01-31 23:26:00,2026-01-31 23:39:16,NaN,1.62,NaN,NaN,237,161,0,17.09,0.00,0.5,0.00,0.0,1.0,21.84,NaN,NaN,0.75
3724885,2,2026-01-31 23:33:53,2026-01-31 23:34:07,NaN,0.00,NaN,NaN,42,42,0,23.19,0.00,0.5,0.00,0.0,1.0,24.69,NaN,NaN,0.00
3724886,2,2026-01-31 23:40:23,2026-01-31 23:56:10,NaN,6.84,NaN,NaN,137,69,0,29.21,0.00,0.5,0.00,0.0,1.0,33.96,NaN,NaN,0.75
3724887,2,2026-01-31 23:10:21,2026-01-31 23:20:00,NaN,1.53,NaN,NaN,137,162,0,19.19,0.00,0.5,0.00,0.0,1.0,23.94,NaN,NaN,0.75


## Chapter 1 : Data Overview

In [3]:
file_path = "data/yellow_tripdata_2026-01.parquet"

trip_duration = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

overview_df = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Total Features",
        "Pickup Date Range",
        "Total Observation Days",
        "Dataset Size (MB)",
        "Exact Duplicate Records",
        "Total Missing Cells",
        "Rows With Missing Values",
        "Negative Fare Records",
        "Non-positive Duration Records",
    ],
    "Value": [
        f"{len(df):,}",
        df.shape[1],
        f"{df['tpep_pickup_datetime'].min():%Y-%m-%d} to "
        f"{df['tpep_pickup_datetime'].max():%Y-%m-%d}",
        df["tpep_pickup_datetime"].dt.date.nunique(),
        round(os.path.getsize(file_path) / (1024 ** 2), 2),
        f"{df.duplicated().sum():,}",
        f"{df.isna().sum().sum():,}",
        f"{df.isna().any(axis=1).sum():,}",
        f"{(df['fare_amount'] < 0).sum():,}",
        f"{(trip_duration <= 0).sum():,}",
    ],
})

display(overview_df)

,Metric,Value
0,Total Records,"3,724,889"
1,Total Features,20
2,Pickup Date Range,2025-12-31 to 2026-02-01
3,Total Observation Days,33
4,Dataset Size (MB),61.19
5,Exact Duplicate Records,0
6,Total Missing Cells,"5,440,290"
7,Rows With Missing Values,"1,088,058"
8,Negative Fare Records,"39,463"
9,Non-positive Duration Records,"45,070"


## Chapter 2 : Data Quality Check

In [4]:
missing_summary = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda data: (data["missing_count"] / len(df) * 100).round(2))
      .query("missing_count > 0")
      .sort_values("missing_count", ascending=False)
)

display(missing_summary)

,missing_count,missing_pct
passenger_count,1088058,29.21
RatecodeID,1088058,29.21
store_and_fwd_flag,1088058,29.21
congestion_surcharge,1088058,29.21
Airport_fee,1088058,29.21


In [5]:
df_clean = df.copy()

df_clean["trip_duration_min"] = (
    df_clean["tpep_dropoff_datetime"] - df_clean["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df_clean = df_clean[
    (df_clean["trip_duration_min"] > 0) &
    (df_clean["fare_amount"] >= 0) &
    (df_clean["total_amount"] >= 0)
].copy()

print(f"Before cleaning: {len(df):,}")
print(f"After cleaning:  {len(df_clean):,}")
print(f"Rows removed:    {len(df) - len(df_clean):,}")

Before cleaning: 3,724,889
After cleaning:  3,639,817
Rows removed:    85,072


### Feature Engineering

In [6]:
# Time features
df_clean["pickup_date"] = df_clean["tpep_pickup_datetime"].dt.date
df_clean["pickup_hour"] = df_clean["tpep_pickup_datetime"].dt.hour
df_clean["pickup_dayofweek"] = df_clean["tpep_pickup_datetime"].dt.dayofweek
df_clean["pickup_day_name"] = df_clean["tpep_pickup_datetime"].dt.day_name()
df_clean["is_weekend"] = (df_clean["pickup_dayofweek"] >= 5).astype(int)

df_clean["is_rush_hour"] = df_clean["pickup_hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)

# EDA-only features
df_clean["speed_mph"] = df_clean["trip_distance"] / (df_clean["trip_duration_min"] / 60)
df_clean["tip_percentage"] = np.where(
    df_clean["fare_amount"] > 0,
    df_clean["tip_amount"] / df_clean["fare_amount"] * 100,
    np.nan
)

df_clean.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,cbd_congestion_fee,trip_duration_min,pickup_date,pickup_hour,pickup_dayofweek,pickup_day_name,is_weekend,is_rush_hour,speed_mph,tip_percentage
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,0.00,5.550000,2026-01-01,0,3,Thursday,0,0,10.486486,50.833333
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,...,0.75,5.716667,2026-01-01,0,3,Thursday,0,0,9.446064,0.000000
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,...,0.75,8.883333,2026-01-01,0,3,Thursday,0,0,9.455910,23.364486
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,...,0.75,42.800000,2026-01-01,0,3,Thursday,0,0,7.822430,28.708010
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,...,0.75,13.500000,2026-01-01,0,3,Thursday,0,0,9.600000,28.518519
